# Lok Sabha Candidate Assets · Dash Dashboard
**DA2402 · Project 3.** Assets declared by Lok Sabha candidates in Tamil Nadu, Maharashtra, Bihar and Uttar Pradesh across 2014, 2019 and 2024, winners against losers.

Run the cells top to bottom. The last cell starts the app inside the notebook.

In [10]:
# !pip install dash plotly pandas dash-bootstrap-components

In [11]:
import os, math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc

print('✅ Libraries imported')

✅ Libraries imported


## Load the merged data
Uses `merged_all_2026-10-20.csv` from the merge notebook (03 v2). It looks in `../clean/`, `./clean/`, the notebook folder and `./merge_v2/`. Set `MERGED_CSV` to point somewhere else.

In [12]:
FNAME = 'merged_all_2026-10-20.csv'
PATHS = [os.environ.get('MERGED_CSV', ''), f'../clean/{FNAME}', f'./clean/{FNAME}', f'./{FNAME}', f'./merge_v2/{FNAME}']
SRC = next(p for p in PATHS if p and os.path.exists(p))

raw = pd.read_csv(SRC)
df = raw[~raw['Is_ByElection']].copy()          # general elections only
df['Cases'] = df['MN_Criminal_Cases']
df['Result'] = np.where(df['Winner'], 'Winner', 'Loser')

YEARS = [2014, 2019, 2024]
STATES = ['Bihar', 'Maharashtra', 'Tamil Nadu', 'Uttar Pradesh']
print(f'{SRC}: {len(df):,} candidate results, {df.groupby(["State","Constituency_No"]).ngroups} seats')

../clean/merged_all_2026-10-20.csv: 10,378 candidate results, 207 seats


## Derived columns
Asset rank inside each seat, criminal-case buckets, education order, age groups, and the person-level pairs used for "same candidate" plots.

In [13]:
# rank of declared assets inside the seat (1 = richest)
df['Asset_Rank'] = df.groupby(['Election_Year', 'State', 'Constituency_No'])['Assets_Rs'].rank(ascending=False, method='min')

CASE_BINS = ['0', '1–2', '3–5', '6+']
df['Case_Bin'] = pd.cut(df['Cases'], [-1, 0, 2, 5, 10_000], labels=CASE_BINS)

EDU = ['Illiterate', 'Literate', '5th Pass', '8th Pass', '10th Pass', '12th Pass',
       'Graduate', 'Graduate Professional', 'Post Graduate', 'Doctorate']
AGE_BINS = ['25–34', '35–44', '45–54', '55–64', '65+']
df['Age_Bin'] = pd.cut(df['MN_Age'].where(df['MN_Age'] >= 25), [24, 34, 44, 54, 64, 120], labels=AGE_BINS)

# consecutive contests by the same person (Person_ID links names across years)
rows = []
for pid, gr in df.dropna(subset=['Person_ID']).sort_values('Election_Year').groupby('Person_ID'):
    recs = gr.drop_duplicates('Election_Year').to_dict('records')
    for a, b in zip(recs, recs[1:]):
        if b['Election_Year'] - a['Election_Year'] != 5:
            continue
        rows.append({'Person_ID': pid, 'State': b['State'], 'Name': b['Candidate'], 'Party': b['Party'],
                     'Year': b['Election_Year'], 'A0': a['Assets_Rs'], 'A1': b['Assets_Rs'],
                     'Move': ('Won' if a['Winner'] else 'Lost') + ' → ' + ('Won' if b['Winner'] else 'Lost'),
                     'dVote': b['Vote_Share_Percentage'] - a['Vote_Share_Percentage']})
pairs = pd.DataFrame(rows)
pairs = pairs[(pairs.A0 > 0) & (pairs.A1 > 0)].copy()
pairs['Ratio'] = pairs.A1 / pairs.A0
MOVES = ['Won → Won', 'Won → Lost', 'Lost → Won', 'Lost → Lost']

# people with an asset figure in 2+ elections, for the candidate picker
repeat = (df[df.Assets_Rs > 0].dropna(subset=['Person_ID'])
          .groupby('Person_ID').filter(lambda g: g.Election_Year.nunique() >= 2))
PEOPLE = (repeat.sort_values('Election_Year').groupby('Person_ID')
          .agg(Name=('Candidate', 'last'), State=('State', 'last'), Max=('Assets_Rs', 'max'))
          .sort_values('Max', ascending=False))
print(f'{len(pairs)} consecutive repeat runs, {len(PEOPLE)} people in the picker')

609 consecutive repeat runs, 537 people in the picker


## Theme
Dark Bootstrap theme (DARKLY), one colour per meaning: winners amber, losers blue, one fixed colour per state.

In [14]:
BG, CARD, GRID, INK, INK2 = '#000000', '#0e0e0e', '#232323', '#ffffff', '#b8b8b8'
WIN, LOSE, GREY, RED = '#c98500', '#3987e5', '#5a5e69', '#e66767'
STATE_C = {'Bihar': '#d95926', 'Maharashtra': '#199e70', 'Tamil Nadu': '#9085e9', 'Uttar Pradesh': '#d55181'}
MOVE_C = {'Won → Won': WIN, 'Won → Lost': '#d95926', 'Lost → Won': '#199e70', 'Lost → Lost': LOSE}
RUPEE_TICKS = dict(tickvals=[3, 4, 5, 6, 7, 8, 9, 10],
                   ticktext=['₹1k', '₹10k', '₹1 lakh', '₹10 lakh', '₹1 cr', '₹10 cr', '₹100 cr', '₹1000 cr'])


def cr(v):
    """Rupees -> '₹9.37 cr' or '₹28 lakh'."""
    if v is None or pd.isna(v):
        return '–'
    return f'₹{v/1e7:.2f} cr' if v >= 1e7 else f'₹{v/1e5:.0f} lakh'


def style(fig, h=420, legend=True):
    fig.update_layout(template='plotly_dark', height=h, showlegend=legend,
                      paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                      font=dict(family='Helvetica Neue, Arial, sans-serif', size=14, color=INK),
                      margin=dict(l=60, r=24, t=30, b=50),
                      legend=dict(orientation='h', x=0, y=1.12, bgcolor='rgba(0,0,0,0)'),
                      hoverlabel=dict(bgcolor='#1c1c1c', font_size=13), uniformtext=dict(minsize=12, mode='show'))
    fig.update_xaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, fixedrange=True)
    fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, fixedrange=True)
    return fig


def graph(fig, gid=None):
    kw = {'id': gid} if gid else {}
    return dcc.Graph(figure=fig, config={'displayModeBar': False}, **kw)


def card(title, body, caption='', md=6):
    """Bold title, a plot (or any component), and a 1–2 line bold caption."""
    if isinstance(body, go.Figure):
        body = graph(body)
    return dbc.Col(dbc.Card(dbc.CardBody([
        html.H4(title, className='card-title fw-bold'),
        body,
        dcc.Markdown(caption, className='cap'),
    ]), className='chart-card h-100'), md=md, className='mb-4')


def section(title):
    return html.H2(title, className='section')


def kpi(value, label, color):
    return dbc.Col(dbc.Card(dbc.CardBody([
        html.Div(value, className='metric-value'),
        html.Div(label, className='metric-label'),
    ]), className='metric-card'), md=3, xs=6, className='mb-4')


def bar_text(fig):
    fig.update_traces(textposition='outside', cliponaxis=False, textfont_size=15)
    return fig

## Figures
One function per tab. Each takes the state-filtered data and returns the tab's layout, so every number in a caption is computed from what is on screen.

In [15]:
def filt(states):
    s = states or STATES
    g = df[df.State.isin(s)]
    return g, g[g.Assets_Rs > 0], pairs[pairs.State.isin(s)]


def medians(a, by):
    return a.groupby(by)['Assets_Rs'].median()

# ---------------- Overview ----------------
def tab_overview(states):
    g, a, p = filt(states)
    m = a.groupby(['Election_Year', 'Winner'])['Assets_Rs'].median().unstack()
    gap = (m[True] / m[False]).round(1)
    grew = (p.Ratio > 1).mean() * 100

    f1 = go.Figure()
    for w, name, c, pos in [(True, 'Winners', WIN, 'top center'), (False, 'Losers', LOSE, 'bottom center')]:
        f1.add_scatter(x=YEARS, y=m[w].values, name=name, mode='lines+markers+text', line=dict(color=c, width=3),
                       marker_size=11, text=[cr(v) for v in m[w]], textposition=pos, textfont_size=14,
                       hovertemplate='%{x}: %{text}<extra>' + name + '</extra>')
    style(f1).update_yaxes(type='log', tickvals=[1e6, 3e6, 1e7, 3e7, 1e8, 3e8],
                           ticktext=['₹10 lakh', '₹30 lakh', '₹1 cr', '₹3 cr', '₹10 cr', '₹30 cr'])
    f1.update_xaxes(tickvals=YEARS, range=[2012.5, 2025.5])

    f2 = go.Figure(go.Bar(x=[str(y) for y in YEARS], y=gap.values, marker_color=WIN, width=0.5,
                          text=[f'{v}×' for v in gap], hovertemplate='%{x}: %{y}×<extra></extra>'))
    bar_text(style(f2, legend=False)).update_yaxes(range=[0, gap.max() * 1.25], ticksuffix='×')

    return [
        html.H1('Do winners get richer, or do the rich win?', className='page-title'),
        html.P([f'{len(g):,} candidates in {g.groupby(["State","Constituency_No"]).ngroups} seats, matched to their sworn asset declarations. '
                f'The median winner went from ', html.B(f'{gap.iloc[0]}×'), ' to ', html.B(f'{gap.iloc[-1]}×'),
                ' the median loser in ten years.'], className='lead-text'),
        dbc.Row([kpi(cr(m.loc[2024, True]), 'Median 2024 winner', WIN),
                 kpi(cr(m.loc[2024, False]), 'Median 2024 loser', LOSE),
                 kpi(f'{gap.loc[2024]}×', 'Winner–loser gap, 2024', WIN),
                 kpi(f'{grew:.0f}%', 'Repeat candidates who got richer', INK)]),
        dbc.Row([
            card('Median declared assets, by election', f1,
                 f'The median winner rose to **{cr(m.loc[2024, True])}**; the median loser barely moved (**{cr(m.loc[2014, False])} → {cr(m.loc[2024, False])}**).'),
            card('How many times richer is the median winner?', f2,
                 f'The gap went from **{gap.iloc[0]}×** in 2014 to **{gap.iloc[-1]}×** in 2024.'),
        ]),
    ]

# ---------------- Winners vs Losers ----------------
EDGES = np.arange(3.0, 10.51, 0.25)

def fig_hist(a, year):
    if year != 'All':
        a = a[a.Election_Year == int(year)]
    f = go.Figure()
    peaks = {}
    for w, name, c in [(False, 'Losers', LOSE), (True, 'Winners', WIN)]:
        v = np.log10(a[a.Winner == w].Assets_Rs).clip(EDGES[0], EDGES[-1] - 1e-9)
        cnt, _ = np.histogram(v, EDGES)
        pct = 100 * cnt / max(cnt.sum(), 1)
        mid = (EDGES[:-1] + EDGES[1:]) / 2
        peaks[w] = 10 ** mid[pct.argmax()]
        f.add_bar(x=mid, y=pct, width=0.23, name=f'{name} ({cnt.sum():,})', marker_color=c, opacity=0.8,
                  customdata=[f'{cr(10**(m-0.125))} – {cr(10**(m+0.125))}' for m in mid],
                  hovertemplate='%{customdata}<br>%{y:.1f}% of ' + name.lower() + '<extra></extra>')
    style(f).update_layout(barmode='overlay')
    # fixed ranges: hiding a group in the legend does not rescale the axes
    f.update_xaxes(range=[3, 10.5], autorange=False, **RUPEE_TICKS)
    f.update_yaxes(range=[0, 22], autorange=False, ticksuffix='%', title='% of group')
    return f, peaks


def tab_wl(states):
    g, a, p = filt(states)
    f_hist, peaks = fig_hist(a, 'All')

    f_box = go.Figure()
    for w, name, c in [(True, 'Winners', WIN), (False, 'Losers', LOSE)]:
        s = a[a.Winner == w]
        f_box.add_box(x=s.Election_Year.astype(str), y=s.Assets_Rs, name=name, marker=dict(color=c, size=3, opacity=0.35),
                      line=dict(color=c, width=2), fillcolor=c + '33', boxpoints='outliers')
    style(f_box).update_layout(boxmode='group')
    f_box.update_yaxes(type='log', range=[3, 10.6], tickvals=[10 ** v for v in RUPEE_TICKS['tickvals']], ticktext=RUPEE_TICKS['ticktext'])
    q1w = a[a.Winner].groupby('Election_Year').Assets_Rs.quantile(0.25)
    medl = a[~a.Winner].groupby('Election_Year').Assets_Rs.median()

    # winner vs runner-up in the same seat
    top2 = g[g.Position <= 2].pivot_table(index=['Election_Year', 'State', 'Constituency_No'], columns='Position', values='Assets_Rs')
    top2 = top2.dropna()
    richer = (top2[1] > top2[2]).groupby(level=0).mean() * 100
    f_ru = go.Figure(go.Bar(x=[str(y) for y in richer.index], y=richer.values, width=0.5, marker_color=WIN,
                            text=[f'{v:.0f}%' for v in richer], hovertemplate='%{x}: %{y:.1f}% of seats<extra></extra>'))
    bar_text(style(f_ru, legend=False)).update_yaxes(range=[0, 100], ticksuffix='%', title='Seats where the winner is richer')

    # how much richer is the winner than the runner-up?
    lr = np.log10(top2[1] / top2[2])
    ticks = [0.01, 0.1, 1, 10, 100]
    f_lb = go.Figure()
    for y in YEARS:
        v = lr.xs(y, level=0)
        f_lb.add_violin(x=[str(y)] * len(v), y=v, line_color=WIN, fillcolor=WIN + '44', points=False, spanmode='hard',
                        box=dict(visible=True, width=0.15, fillcolor=BG, line_color=INK), bandwidth=0.2, hoverinfo='skip')
    style(f_lb, legend=False).update_yaxes(range=[-2.3, 2.3], autorange=False, tickvals=[math.log10(t) for t in ticks],
                                           ticktext=[f'{t:g}×' for t in ticks], title='Winner ÷ runner-up assets')
    f_lb.add_hline(y=0, line=dict(color='#8a8d96', dash='dash', width=1.5))
    medr = 10 ** lr.groupby(level=0).median()

    return [
        html.H1('Winners vs Losers', className='page-title'),
        section('The wealth gap'),
        dbc.Row([
            card('Where candidates sit on the wealth scale', html.Div([
                dbc.RadioItems(id='hist-year', options=['All'] + [str(y) for y in YEARS], value='All', inline=True, className='mb-2'),
                graph(f_hist, 'hist-graph')]),
                f'Losers peak near **{cr(peaks[False])}**; winners peak near **{cr(peaks[True])}**. The axes stay fixed when you toggle the legend.'),
            card('Spread of assets in each election', f_box,
                 f'The bottom quarter of 2024 winners (**{cr(q1w.loc[2024])}**) is richer than the median loser (**{cr(medl.loc[2024])}**).'),
        ]),
        section('Head to head'),
        dbc.Row([
            card('Is the winner richer than the runner-up?', f_ru,
                 f'In 2024 the winner out-declared the runner-up in **{richer.loc[2024]:.0f}%** of seats.'),
            card('How much richer is the winner than the runner-up?', f_lb,
                 f'Above the line = winner richer. The typical 2024 winner held **{medr.loc[2024]:.1f}×** the runner-up\'s assets.'),
        ]),
    ]

# ---------------- States & seats ----------------
def state_lines(a, w, unit, scale):
    m = a[a.Winner == w].groupby(['State', 'Election_Year']).Assets_Rs.median().unstack() / scale
    f = go.Figure()
    for s in m.index:
        f.add_scatter(x=YEARS, y=m.loc[s].values, name=s, mode='lines+markers', line=dict(color=STATE_C[s], width=3), marker_size=10,
                      hovertemplate=f'{s} %{{x}}: ₹%{{y:.2f}} {unit}<extra></extra>')
    # end labels, nudged apart so they never overlap
    ends = m[2024].sort_values()
    top = m.max().max() * 1.12
    pos, last = [], -1e9
    for v in ends:
        last = max(v, last + top * 0.06)
        pos.append(last)
    for (s, v), y in zip(ends.items(), pos):
        f.add_annotation(x=2024, y=y, xanchor='left', xshift=10, showarrow=False, text=f'₹{v:.1f} {unit}', font_size=13)
    style(f).update_xaxes(tickvals=YEARS, range=[2013, 2026.6])
    f.update_yaxes(range=[0, max(pos[-1], top) * 1.05])
    return f, m


def tab_states(states):
    g, a, p = filt(states)
    f_w, mw = state_lines(a, True, 'cr', 1e7)
    f_l, ml = state_lines(a, False, 'L', 1e5)

    top = a[a.Winner].nlargest(15, 'Assets_Rs').iloc[::-1]
    f_top = go.Figure(go.Bar(y=[f'{n.title()} · {pa} · {y}' for n, pa, y in zip(top.Candidate, top.Party, top.Election_Year)],
                             x=top.Assets_Rs / 1e7, orientation='h', marker_color=[STATE_C[s] for s in top.State],
                             text=[f'₹{v/1e7:.0f} cr' for v in top.Assets_Rs], customdata=top.Constituency_Name.str.title(),
                             hovertemplate='%{y}<br>%{customdata}: ₹%{x:.1f} cr<extra></extra>'))
    bar_text(style(f_top, h=520, legend=False)).update_layout(margin=dict(l=10))
    f_top.update_yaxes(tickfont_size=12, automargin=True)
    f_top.update_xaxes(title='₹ crore', range=[0, top.Assets_Rs.max() / 1e7 * 1.18])

    t = a.groupby(['Constituency_Type', 'Winner']).Assets_Rs.median().unstack().reindex(['GEN', 'SC', 'ST']).dropna()
    f_t = go.Figure()
    for w, name, c in [(True, 'Winners', WIN), (False, 'Losers', LOSE)]:
        f_t.add_bar(x=t.index, y=t[w] / 1e7, name=name, marker_color=c, text=[cr(v) for v in t[w]],
                    hovertemplate='%{x}: ₹%{y:.2f} cr<extra>' + name + '</extra>')
    bar_text(style(f_t, h=520)).update_layout(barmode='group', bargap=0.3)
    f_t.update_yaxes(title='Median assets (₹ crore)', range=[0, t.max().max() / 1e7 * 1.25])

    return [
        html.H1('States & Seats', className='page-title'),
        section('By state'),
        dbc.Row([
            card('Median winner, by state (₹ crore)', f_w,
                 f'**{mw[2024].idxmax()}** has the richest winners (**₹{mw[2024].max():.2f} cr** in 2024); **{mw[2024].idxmin()}** the least (**₹{mw[2024].min():.2f} cr**).'),
            card('Median loser, by state (₹ lakh)', f_l,
                 f'Losers stay in lakhs: **{ml[2024].idxmax()}** highest at **₹{ml[2024].max():.1f} lakh**, **{ml[2024].idxmin()}** lowest at **₹{ml[2024].min():.1f} lakh**.'),
        ]),
        section('Candidates and seat types'),
        dbc.Row([
            card('The 15 richest winning declarations', f_top,
                 f'**{top.iloc[-1].Candidate.title()}** ({top.iloc[-1].Party}, {top.iloc[-1].Election_Year}) tops the list at **{cr(top.iloc[-1].Assets_Rs)}**. Colour = state.'),
            card('Reserved (SC/ST) vs general seats', f_t,
                 f'Winners in general seats declare **{cr(t.loc["GEN", True])}**; in SC seats **{cr(t.loc["SC", True])}**.'
                 + (f' ST: **{cr(t.loc["ST", True])}**.' if 'ST' in t.index else '')),
        ]),
    ]

# ---------------- Parties ----------------
def mwu_p(x, y):
    """Two-sided Mann-Whitney U p-value (normal approximation with tie correction)."""
    x, y = np.asarray(x), np.asarray(y)
    n1, n2 = len(x), len(y)
    allv = np.concatenate([x, y])
    r = pd.Series(allv).rank().values
    u = r[:n1].sum() - n1 * (n1 + 1) / 2
    _, t = np.unique(allv, return_counts=True)
    n = n1 + n2
    sd = math.sqrt(n1 * n2 / 12 * ((n + 1) - (t ** 3 - t).sum() / (n * (n - 1))))
    return max(math.erfc(abs((u - n1 * n2 / 2) / sd) / math.sqrt(2)), 1e-300)


def party_table(a):
    rows = []
    for party, gr in a.groupby('Party'):
        if len(gr) < 20 or party == 'IND':
            continue
        rest = a[a.Party != party].Assets_Rs
        rows.append({'Party': party, 'n': len(gr), 'wins': int(gr.Winner.sum()), 'med': gr.Assets_Rs.median(),
                     'medW': gr[gr.Winner].Assets_Rs.median(), 'medL': gr[~gr.Winner].Assets_Rs.median(),
                     'fc': math.log2(gr.Assets_Rs.median() / rest.median()), 'p': -math.log10(mwu_p(gr.Assets_Rs, rest)),
                     'group': gr.Party_Group.mode().iloc[0]})
    t = pd.DataFrame(rows)
    t['strike'] = 100 * t.wins / t.n
    return t


def fig_party_box(a, party):
    s = a[a.Party == party]
    f = go.Figure()
    for w, name, c in [(True, 'Winners', WIN), (False, 'Losers', LOSE)]:
        x = s[s.Winner == w]
        f.add_box(x=x.Election_Year.astype(str), y=x.Assets_Rs, name=f'{name} ({len(x)})', marker=dict(color=c, size=4),
                  line=dict(color=c, width=2), fillcolor=c + '33', boxpoints='all', jitter=0.4, pointpos=0,
                  customdata=x.Candidate.str.title(), hovertemplate='%{customdata}: %{y:,.0f}<extra></extra>')
    style(f).update_layout(boxmode='group')
    f.update_xaxes(categoryorder='array', categoryarray=[str(y) for y in YEARS])
    f.update_yaxes(type='log', range=[4, 10.6], tickvals=[10 ** v for v in RUPEE_TICKS['tickvals']], ticktext=RUPEE_TICKS['ticktext'])
    return f


def tab_parties(states):
    g, a, p = filt(states)
    t = party_table(a)

    d = t[t.wins >= 5].dropna(subset=['medW', 'medL']).sort_values('medW')
    f_db = go.Figure()
    for _, r in d.iterrows():
        f_db.add_scatter(x=[r.medL / 1e7, r.medW / 1e7], y=[r.Party] * 2, mode='lines', line=dict(color=GREY, width=3),
                         showlegend=False, hoverinfo='skip')
    f_db.add_scatter(x=d.medL / 1e7, y=d.Party, mode='markers', name='Losers', marker=dict(color=LOSE, size=13),
                     hovertemplate='%{y} losers: ₹%{x:.2f} cr<extra></extra>')
    f_db.add_scatter(x=d.medW / 1e7, y=d.Party, mode='markers', name='Winners', marker=dict(color=WIN, size=13),
                     hovertemplate='%{y} winners: ₹%{x:.2f} cr<extra></extra>')
    style(f_db, h=460).update_xaxes(type='log', title='Median assets (₹ crore, log)', tickvals=[0.1, 0.3, 1, 3, 10, 30],
                                    ticktext=['0.1', '0.3', '1', '3', '10', '30'])

    gc = {'Main party': WIN, 'Other party': '#9085e9'}
    f_v = go.Figure()
    lab = set(t.nlargest(9, 'p').Party) | set(t.nsmallest(1, 'fc').Party)
    for grp, s in t.groupby('group'):
        f_v.add_scatter(x=s.fc, y=s.p.clip(upper=170), mode='markers+text', name=grp,
                        text=[x if x in lab else '' for x in s.Party], textposition='top center', textfont_size=12,
                        marker=dict(color=gc.get(grp, GREY), size=np.clip(np.sqrt(s.n) * 1.2, 8, None), opacity=0.85, line=dict(width=1, color=CARD)),
                        customdata=np.c_[s.Party, s.n, (s.med / 1e7).round(2), s.wins],
                        hovertemplate='%{customdata[0]}: %{customdata[1]} candidates, %{customdata[3]} wins<br>median ₹%{customdata[2]} cr<extra></extra>')
    style(f_v, h=460).update_xaxes(title='log₂ (party median ÷ everyone else)', zeroline=True, zerolinecolor='#8a8d96')
    f_v.update_yaxes(title='−log₁₀ p (Mann–Whitney)')
    bjp = t.set_index('Party').loc['BJP'] if 'BJP' in set(t.Party) else t.nlargest(1, 'p').iloc[0]

    f_s = go.Figure()
    for grp, s in t.groupby('group'):
        f_s.add_scatter(x=s.med / 1e7, y=s.strike, mode='markers+text', name=grp, text=[x if x in lab else '' for x in s.Party],
                        textposition='top center', textfont_size=12,
                        marker=dict(color=gc.get(grp, GREY), size=np.clip(np.sqrt(s.n) * 1.2, 8, None), opacity=0.85, line=dict(width=1, color=CARD)),
                        customdata=np.c_[s.Party, s.n, s.wins],
                        hovertemplate='%{customdata[0]}: %{customdata[2]} of %{customdata[1]} won<extra></extra>')
    style(f_s, h=460).update_xaxes(type='log', title='Median candidate assets (₹ crore, log)', tickvals=[0.01, 0.1, 1, 10],
                                   ticktext=['0.01', '0.1', '1', '10'])
    f_s.update_yaxes(title='Strike rate (% of candidates who won)', ticksuffix='%')
    corr = np.corrcoef(np.log10(t.med), t.strike)[0, 1]

    big = t.sort_values('n', ascending=False).Party.tolist()
    default = 'BJP' if 'BJP' in big else big[0]
    return [
        html.H1('Parties', className='page-title'),
        section('Rich parties, rich winners'),
        dbc.Row([
            card('Winners vs losers inside each party', f_db,
                 f'Winners out-declare their own party\'s losers in **{(d.medW > d.medL).sum()} of {len(d)}** parties (5+ wins). **{d.iloc[-1].Party}** winners are the richest (**{cr(d.iloc[-1].medW)}**).'),
            card('Volcano: which parties field richer candidates?', f_v,
                 f'Top right = much richer and certain. **{bjp.name}** candidates\' median is **{2 ** bjp.fc:.0f}×** everyone else\'s. Parties with 20+ candidates.'),
        ]),
        section('Explore a party'),
        dbc.Row([
            card('Richer parties win more', f_s,
                 f'Party median wealth and strike rate move together (correlation **{corr:.2f}**, log assets).'),
            card('Assets of one party\'s candidates', html.Div([
                dbc.Select(id='party-pick', options=[{'label': x, 'value': x} for x in big], value=default, className='mb-2'),
                graph(fig_party_box(a, default), 'party-graph')]),
                'Pick a party. Each dot is one candidate; hover for the name.'),
        ]),
    ]

# ---------------- Same candidates ----------------
def fig_person(pid):
    s = df[(df.Person_ID == pid) & (df.Assets_Rs > 0)].sort_values('Election_Year')
    f = go.Figure(go.Scatter(x=s.Election_Year, y=s.Assets_Rs / 1e7, mode='lines+markers+text',
                             line=dict(color=INK2, width=2), marker=dict(size=16, color=[WIN if w else LOSE for w in s.Winner],
                                                                         line=dict(width=2, color=CARD)),
                             text=[f'{cr(v)} · {"Won" if w else "Lost"} ({pa})' for v, w, pa in zip(s.Assets_Rs, s.Winner, s.Party)],
                             textposition='top center', hovertemplate='%{x}: %{text}<extra></extra>'))
    style(f, legend=False).update_xaxes(tickvals=YEARS, range=[2012, 2026])
    f.update_yaxes(title='Declared assets (₹ crore)', rangemode='tozero', range=[0, s.Assets_Rs.max() / 1e7 * 1.3])
    return f


def tab_same(states):
    g, a, p = filt(states)

    f_sc = go.Figure()
    f_sc.add_scatter(x=[0.001, 5000], y=[0.001, 5000], mode='lines', line=dict(color='#8a8d96', dash='dash'), name='No change', hoverinfo='skip')
    for mv in MOVES:
        s = p[p.Move == mv]
        f_sc.add_scatter(x=s.A0 / 1e7, y=s.A1 / 1e7, mode='markers', name=f'{mv} ({len(s)})',
                         marker=dict(color=MOVE_C[mv], size=8, opacity=0.8, line=dict(width=1, color=CARD)),
                         text=s.Name.str.title() + ' (' + s.Party + ', ' + s.Year.astype(str) + ')',
                         hovertemplate='%{text}<br>₹%{x:.2f} cr → ₹%{y:.2f} cr<extra></extra>')
    ax = dict(type='log', range=[-2.2, 3.3], tickvals=[0.01, 0.1, 1, 10, 100, 1000], ticktext=['0.01', '0.1', '1', '10', '100', '1000'])
    style(f_sc, h=470).update_xaxes(title='Earlier election (₹ crore)', **ax)
    f_sc.update_yaxes(title='Next election (₹ crore)', **ax)
    f_sc.update_layout(legend=dict(y=1.16))

    grew = p.groupby('Move').Ratio.apply(lambda r: 100 * (r > 1).mean()).reindex(MOVES)
    n = p.Move.value_counts().reindex(MOVES).fillna(0).astype(int)
    f_g = go.Figure(go.Bar(x=MOVES, y=grew.values, width=0.55, marker_color=[MOVE_C[m] for m in MOVES],
                           text=[f'{v:.0f}%' for v in grew], customdata=n.values, hovertemplate='%{x}: %{y:.0f}% grew (n=%{customdata})<extra></extra>'))
    bar_text(style(f_g, h=470, legend=False)).update_yaxes(range=[0, 110], ticksuffix='%')

    ticks = [0.1, 0.25, 0.5, 1, 2, 4, 10, 25]
    med = p.groupby('Move').Ratio.median().reindex(MOVES)
    f_vi = go.Figure()
    for mv in MOVES:
        s = np.log10(p[p.Move == mv].Ratio)
        f_vi.add_violin(x=[mv] * len(s), y=s, name=mv, line_color=MOVE_C[mv], fillcolor=MOVE_C[mv] + '55', points=False,
                        box=dict(visible=True, width=0.18, fillcolor=BG, line_color=INK), spanmode='hard', bandwidth=0.08,
                        hoverinfo='skip')
        f_vi.add_annotation(x=mv, y=math.log10(32), yanchor='top', showarrow=False, text=f'median {med[mv]:.2f}×', font_size=14)
    style(f_vi, h=470, legend=False).update_yaxes(range=[math.log10(0.08), math.log10(32)], autorange=False,
                                                  tickvals=[math.log10(t) for t in ticks], ticktext=[f'{t}×' for t in ticks],
                                                  title='Assets next time ÷ assets before')
    f_vi.add_hline(y=0, line=dict(color='#8a8d96', dash='dash', width=1.5))

    three = g[g.Person_ID.map(g.dropna(subset=['Person_ID']).groupby('Person_ID').Election_Year.nunique()) == 3]
    wins = three.groupby('Person_ID').Winner.sum()
    grp = wins.map(lambda k: 'Won all 3' if k == 3 else ('Never won' if k == 0 else 'Won 1 or 2'))
    three = three.assign(G=three.Person_ID.map(grp))
    t3 = three[three.Assets_Rs > 0].groupby(['G', 'Election_Year']).Assets_Rs.median().unstack()
    f3 = go.Figure()
    for k, c, pos in [('Won all 3', WIN, 'top center'), ('Won 1 or 2', '#199e70', 'bottom center'), ('Never won', LOSE, 'top center')]:
        if k not in t3.index:
            continue
        f3.add_scatter(x=YEARS, y=t3.loc[k].values / 1e7, name=f'{k} ({(grp == k).sum()})', mode='lines+markers+text', line=dict(color=c, width=3),
                       marker_size=11, text=[cr(v) if (i == 2 or (i == 0 and k != 'Won 1 or 2')) else '' for i, v in enumerate(t3.loc[k])], textposition=pos, textfont_size=13,
                       hovertemplate=k + ' %{x}: ₹%{y:.2f} cr<extra></extra>')
    style(f3, h=470).update_yaxes(type='log', tickvals=[0.1, 0.3, 1, 3, 10, 30], ticktext=['₹10 lakh', '₹30 lakh', '₹1 cr', '₹3 cr', '₹10 cr', '₹30 cr'])
    f3.update_xaxes(tickvals=YEARS, range=[2012.5, 2025.5])

    opts = [{'label': f'{r.Name.title()} ({r.State})', 'value': str(pid)} for pid, r in PEOPLE[PEOPLE.State.isin(states or STATES)].iterrows()]
    first = opts[0]['value']
    cap3 = ''
    if {'Won all 3', 'Never won'} <= set(t3.index):
        cap3 = (f'The gap is where they start: **{cr(t3.loc["Won all 3", 2014])} → {cr(t3.loc["Won all 3", 2024])}** for three-time winners '
                f'vs **{cr(t3.loc["Never won", 2014])} → {cr(t3.loc["Never won", 2024])}** for never-winners.')
    return [
        html.H1('Same Candidates Over Time', className='page-title'),
        section('Did they get richer?'),
        dbc.Row([
            card('Assets at one election vs the next', f_sc,
                 f'**{100 * (p.Ratio > 1).mean():.0f}%** of {len(p)} repeat runs sit above the line: they declared more the next time.'),
            card('Share whose assets grew, by result', f_g,
                 f'Re-elected winners grew most often (**{grew["Won → Won"]:.0f}%**), repeat losers least (**{grew["Lost → Lost"]:.0f}%**).'),
        ]),
        section('How fast?'),
        dbc.Row([
            card('Growth between elections (log scale)', f_vi,
                 f'Same pace for all: the typical repeat candidate grows **{p.Ratio.median():.2f}×** per election, won or lost.'),
            card('People who ran all three times', f3, cap3),
        ]),
        section('Look up a candidate'),
        dbc.Row([
            card('One person across elections', html.Div([
                dbc.Select(id='person-pick', options=opts, value=first, className='mb-2'),
                graph(fig_person(float(first)), 'person-graph')]),
                'Pick anyone who contested more than once (richest first). Amber = won, blue = lost.', md=12),
        ]),
    ]

# ---------------- Criminal cases ----------------
def tab_crime(states):
    g, a, p = filt(states)
    c = g[g.Cases.notna()]

    win_all = c.groupby('Case_Bin', observed=True).Winner.mean() * 100
    win_main = c[c.Party_Group == 'Main party'].groupby('Case_Bin', observed=True).Winner.mean() * 100
    f_w = go.Figure()
    f_w.add_bar(x=CASE_BINS, y=win_all.values, name='All candidates', marker_color=GREY, text=[f'{v:.1f}%' for v in win_all])
    f_w.add_bar(x=CASE_BINS, y=win_main.values, name='Main-party candidates', marker_color=RED, text=[f'{v:.1f}%' for v in win_main])
    bar_text(style(f_w)).update_layout(barmode='group', bargap=0.3)
    f_w.update_xaxes(type='category', title='Declared criminal cases')
    f_w.update_yaxes(range=[0, max(win_main.max(), win_all.max()) * 1.2], ticksuffix='%')

    ca = c[c.Assets_Rs > 0]
    medc = ca.groupby('Case_Bin', observed=True).Assets_Rs.median()
    f_v = go.Figure()
    for b in CASE_BINS:
        s = np.log10(ca[ca.Case_Bin == b].Assets_Rs)
        f_v.add_violin(x=[f'{b} cases'] * len(s), y=s, line_color=RED, fillcolor=RED + '44', points=False,
                       box=dict(visible=True, width=0.18, fillcolor=BG, line_color=INK), spanmode='hard', bandwidth=0.15, hoverinfo='skip')
        f_v.add_annotation(x=f'{b} cases', y=10.3, yanchor='top', showarrow=False, text=f'median {cr(medc[b])}', font_size=13)
    style(f_v, legend=False).update_yaxes(range=[4, 10.3], autorange=False, tickvals=[5, 6, 7, 8, 9, 10],
                                          ticktext=['₹1 lakh', '₹10 lakh', '₹1 cr', '₹10 cr', '₹100 cr', '₹1000 cr'])

    av = c.groupby(['Election_Year', 'Winner']).Cases.mean().unstack()
    anyc = c.groupby(['Election_Year', 'Winner']).Cases.apply(lambda s: 100 * (s > 0).mean()).unstack()
    f_a = go.Figure()
    for w, name, col in [(True, 'Winners', WIN), (False, 'Losers', LOSE)]:
        f_a.add_bar(x=[str(y) for y in YEARS], y=av[w].values, name=name, marker_color=col, text=[f'{v:.1f}' for v in av[w]],
                    customdata=anyc[w].values, hovertemplate='%{x}: %{y:.2f} cases<br>%{customdata:.0f}% have at least one<extra>' + name + '</extra>')
    bar_text(style(f_a)).update_layout(barmode='group', bargap=0.35)
    f_a.update_yaxes(title='Cases per candidate', range=[0, av.max().max() * 1.25])

    big = c.Party.value_counts()
    big = big[(big >= 40) & ~big.index.isin(['IND', 'OTHER'])].index
    sh = (c[c.Party.isin(big)].groupby('Party').Cases.apply(lambda s: 100 * (s > 0).mean()).sort_values())
    f_p = go.Figure(go.Bar(y=sh.index, x=sh.values, orientation='h', marker_color=[RED if v >= 50 else GREY for v in sh],
                           text=[f'{v:.0f}%' for v in sh], hovertemplate='%{y}: %{x:.0f}% declared a case<extra></extra>'))
    bar_text(style(f_p, legend=False)).update_xaxes(range=[0, 100], ticksuffix='%')
    ind = 100 * (c[c.Party == 'IND'].Cases > 0).mean()

    return [
        html.H1('Criminal Cases', className='page-title'),
        section('Cases and winning'),
        dbc.Row([
            card('Criminal cases vs win rate', f_w,
                 f'More cases, more wins: **{win_all["6+"]:.1f}%** of candidates with 6+ cases won vs **{win_all["0"]:.1f}%** with none. It holds inside main parties too.'),
            card('Average criminal cases: winners vs losers', f_a,
                 f'In 2024 winners averaged **{av.loc[2024, True]:.2f}** cases vs **{av.loc[2024, False]:.2f}** for losers; **{anyc.loc[2024, True]:.0f}%** of winners declared one.'),
        ]),
        section('Cases, money and parties'),
        dbc.Row([
            card('Criminal cases vs assets', f_v,
                 f'Candidates with cases are richer: median **{cr(medc["6+"])}** with 6+ cases vs **{cr(medc["0"])}** with none.'),
            card('Share of candidates with at least one case, by party', f_p,
                 f'**{sh.idxmax()}** leads at **{sh.max():.0f}%**; independents sit at **{ind:.0f}%**. Parties with 40+ candidates.'),
        ]),
    ]

# ---------------- Education & age ----------------
def tab_people(states):
    g, a, p = filt(states)
    e = a[a.MN_Education.isin(EDU)]
    me = e.groupby('MN_Education').Assets_Rs.median().reindex(EDU)
    we = g[g.MN_Education.isin(EDU)].groupby('MN_Education').Winner.mean().reindex(EDU) * 100
    f_e = go.Figure(go.Bar(y=EDU, x=me.values / 1e5, orientation='h', marker_color=LOSE, text=[cr(v) for v in me],
                           hovertemplate='%{y}: ₹%{x:.1f} lakh<extra></extra>'))
    bar_text(style(f_e, h=460, legend=False)).update_xaxes(type='log', title='Median assets (₹ lakh, log)', range=[0, math.log10(me.max() / 1e5) + 0.4], tickvals=[1, 10, 100], ticktext=['₹1 lakh', '₹10 lakh', '₹1 cr'])
    f_ew = go.Figure(go.Bar(y=EDU, x=we.values, orientation='h', marker_color=WIN, text=[f'{v:.1f}%' for v in we],
                            hovertemplate='%{y}: %{x:.1f}% won<extra></extra>'))
    bar_text(style(f_ew, h=460, legend=False)).update_xaxes(range=[0, we.max() * 1.25], ticksuffix='%', title='Win rate')

    ag = a[a.Age_Bin.notna()]
    ma = ag.groupby(['Age_Bin', 'Winner'], observed=True).Assets_Rs.median().unstack()
    f_a = go.Figure()
    for w, name, c in [(True, 'Winners', WIN), (False, 'Losers', LOSE)]:
        f_a.add_bar(x=AGE_BINS, y=ma[w].values / 1e7, name=name, marker_color=c, text=[cr(v) for v in ma[w]],
                    hovertemplate='%{x}: ₹%{y:.2f} cr<extra>' + name + '</extra>')
    bar_text(style(f_a)).update_layout(barmode='group', bargap=0.3)
    f_a.update_yaxes(title='Median assets (₹ crore)', range=[0, ma.max().max() / 1e7 * 1.25])

    gg = g[g.MN_Age >= 25]
    f_ad = go.Figure()
    for w, name, c in [(True, 'Winners', WIN), (False, 'Losers', LOSE)]:
        s = gg[gg.Winner == w].MN_Age
        f_ad.add_violin(x=[str(y) for y in gg[gg.Winner == w].Election_Year], y=s, name=name, line_color=c, fillcolor=c + '44',
                        side='positive' if w else 'negative', points=False, meanline_visible=True, hoverinfo='skip', scalemode='width')
    style(f_ad).update_layout(violinmode='overlay', violingap=0)
    f_ad.update_yaxes(title='Age', range=[20, 90])
    f_ad.update_xaxes(type='category', range=[-0.6, 2.6])
    agew = gg.groupby('Winner').MN_Age.median()

    return [
        html.H1('Education & Age', className='page-title'),
        section('Education'),
        dbc.Row([
            card('Median assets by education', f_e,
                 f'Doctorates declare **{cr(me["Doctorate"])}**; illiterate candidates **{cr(me["Illiterate"])}**.'),
            card('Win rate by education', f_ew,
                 f'**{we["Doctorate"]:.1f}%** of doctorates won vs **{we["Illiterate"]:.1f}%** of illiterate candidates.'),
        ]),
        section('Age'),
        dbc.Row([
            card('Median assets by age group', f_a,
                 f'Wealth peaks in the **{ma[True].idxmax()}** group for winners (**{cr(ma[True].max())}**).'),
            card('Age of winners vs losers', f_ad,
                 f'The median winner is **{agew[True]:.0f}**, the median loser **{agew[False]:.0f}**.'),
        ]),
    ]

# ---------------- Women ----------------
def tab_women(states):
    g, a, p = filt(states)
    gs = g[g.Sex.isin(['M', 'F'])]
    yrs = sorted(gs.Election_Year.unique())  # the 2024 results file has no gender column
    ly = yrs[-1]
    cnt = gs.groupby(['Election_Year', 'Sex']).size().unstack()
    share = 100 * cnt['F'] / cnt.sum(axis=1)
    wins = gs[gs.Winner].groupby(['Election_Year', 'Sex']).size().unstack().fillna(0)
    wshare = 100 * wins['F'] / wins.sum(axis=1)

    f_s = go.Figure()
    f_s.add_bar(x=[str(y) for y in yrs], y=share.values, name='Women among candidates', marker_color='#9085e9', text=[f'{v:.1f}%' for v in share])
    f_s.add_bar(x=[str(y) for y in yrs], y=wshare.values, name='Women among winners', marker_color=WIN, text=[f'{v:.1f}%' for v in wshare])
    bar_text(style(f_s)).update_layout(barmode='group', bargap=0.3)
    f_s.update_yaxes(ticksuffix='%', range=[0, max(share.max(), wshare.max()) * 1.3])

    wr = gs.groupby(['Election_Year', 'Sex']).Winner.mean().unstack() * 100
    f_r = go.Figure()
    for s, name, c in [('F', 'Women', '#9085e9'), ('M', 'Men', GREY)]:
        f_r.add_scatter(x=yrs, y=wr[s].values, name=name, mode='lines+markers+text', line=dict(color=c, width=3), marker_size=11,
                        text=[f'{v:.1f}%' for v in wr[s]], textposition='top center')
    style(f_r).update_xaxes(tickvals=yrs, range=[yrs[0] - 1.5, yrs[-1] + 1.5])
    f_r.update_yaxes(ticksuffix='%', range=[0, wr.max().max() * 1.35], title='Win rate')

    am = a[a.Sex.isin(['M', 'F'])].groupby(['Sex', 'Winner']).Assets_Rs.median().unstack()
    f_m = go.Figure()
    for w, name, c in [(True, 'Winners', WIN), (False, 'Losers', LOSE)]:
        f_m.add_bar(x=['Women', 'Men'], y=[am.loc['F', w] / 1e7, am.loc['M', w] / 1e7], name=name, marker_color=c,
                    text=[cr(am.loc['F', w]), cr(am.loc['M', w])])
    bar_text(style(f_m)).update_layout(barmode='group', bargap=0.3)
    f_m.update_yaxes(title='Median assets (₹ crore)', range=[0, am.max().max() / 1e7 * 1.25])

    ps = gs[gs.Party_Group == 'Main party'].groupby('Sex').Winner.mean() * 100
    f_pg = go.Figure()
    grp = gs.groupby(['Party_Group', 'Sex']).size().unstack().fillna(0)
    shp = (100 * grp['F'] / grp.sum(axis=1)).reindex(['Main party', 'Other party', 'Independent'])
    f_pg.add_bar(x=shp.index, y=shp.values, marker_color='#9085e9', text=[f'{v:.1f}%' for v in shp], width=0.5)
    bar_text(style(f_pg, legend=False)).update_yaxes(ticksuffix='%', range=[0, shp.max() * 1.3], title='Share of candidates who are women')

    return [
        html.H1('Women Candidates', className='page-title'),
        html.P('Gender is only recorded for 2014 and 2019; the 2024 results file has no gender column.', className='lead-text'),
        section('How many?'),
        dbc.Row([
            card('Women among candidates and winners', f_s,
                 f'Women were **{share.loc[ly]:.1f}%** of {ly} candidates but **{wshare.loc[ly]:.1f}%** of winners.'),
            card('Win rate: women vs men', f_r,
                 f'Inside main parties, **{ps.get("F", 0):.0f}%** of women won vs **{ps.get("M", 0):.0f}%** of men.'),
        ]),
        section('Money and tickets'),
        dbc.Row([
            card('Median assets: women vs men', f_m,
                 f'Women winners declare **{cr(am.loc["F", True])}**, men winners **{cr(am.loc["M", True])}**.'),
            card('Where women get tickets', f_pg,
                 f'**{shp.idxmax()}** candidates are the most often women (**{shp.max():.1f}%**).'),
        ]),
    ]

# ---------------- Experimental ----------------
def tab_exp(states):
    g, a, p = filt(states)
    rk = g[g.Asset_Rank.notna()]
    wr = rk.groupby(rk.Asset_Rank.clip(upper=6)).Winner.mean() * 100
    labels = ['Richest', '2nd', '3rd', '4th', '5th', '6th or lower']
    f_r = go.Figure(go.Bar(x=labels, y=wr.values, width=0.6, marker_color=[WIN] + [GREY] * 5, text=[f'{v:.1f}%' for v in wr],
                           hovertemplate='%{x}: %{y:.1f}% won<extra></extra>'))
    bar_text(style(f_r, legend=False)).update_yaxes(range=[0, wr.max() * 1.25], ticksuffix='%')

    f_vs = go.Figure()
    for w, name, c, sz, op in [(False, 'Losers', LOSE, 5, 0.35), (True, 'Winners', WIN, 7, 0.85)]:
        s = a[a.Winner == w]
        f_vs.add_scattergl(x=np.log10(s.Assets_Rs), y=s.Vote_Share_Percentage, mode='markers', name=name,
                           marker=dict(color=c, size=sz, opacity=op), text=s.Candidate.str.title(),
                           hovertemplate='%{text}: %{y}% of votes<extra></extra>')
    style(f_vs).update_xaxes(range=[3, 10.5], **RUPEE_TICKS)
    f_vs.update_yaxes(range=[0, 80], ticksuffix='%', title='Vote share')
    rho = a.Assets_Rs.rank().corr(a.Vote_Share_Percentage.rank())

    ticks = [0.1, 0.5, 1, 2, 10]
    f_gn = go.Figure()
    for mv in MOVES:
        s = p[p.Move == mv]
        f_gn.add_scattergl(x=np.log10(s.Ratio), y=s.dVote, mode='markers', name=mv, marker=dict(color=MOVE_C[mv], size=7, opacity=0.8),
                           text=s.Name.str.title(), hovertemplate='%{text}: %{y:+.1f} pts<extra></extra>')
    style(f_gn).update_xaxes(range=[-1.3, 1.6], tickvals=[math.log10(t) for t in ticks], ticktext=[f'{t}×' for t in ticks], title='Asset growth (log)')
    f_gn.update_yaxes(range=[-40, 40], ticksuffix=' pts', title='Vote share change')
    r_gain = np.corrcoef(np.log10(p.Ratio), p.dVote.fillna(0))[0, 1]

    # fate of 2014 winners (Sankey)
    pid14 = g[(g.Election_Year == 2014) & g.Winner].Person_ID.dropna().unique()
    def st(pid, y):
        s = g[(g.Person_ID == pid) & (g.Election_Year == y)]
        return 'Not traced' if s.empty else ('Won' if s.Winner.any() else 'Lost')
    fate = pd.DataFrame({'s19': [st(x, 2019) for x in pid14], 's24': [st(x, 2024) for x in pid14]})
    S = ['Won', 'Lost', 'Not traced']
    nodes = ['2014 winners'] + [f'{s} 2019' for s in S] + [f'{s} 2024' for s in S]
    col = lambda n: WIN if n.startswith(('Won', '2014')) else (LOSE if n.startswith('Lost') else GREY)
    src, tgt, val = [], [], []
    for s in S:
        src.append(0); tgt.append(nodes.index(f'{s} 2019')); val.append(int((fate.s19 == s).sum()))
        for s2 in S:
            src.append(nodes.index(f'{s} 2019')); tgt.append(nodes.index(f'{s2} 2024')); val.append(int(((fate.s19 == s) & (fate.s24 == s2)).sum()))
    tot = {n: sum(v for t_, v in zip(tgt, val) if t_ == i) for i, n in enumerate(nodes)}
    tot['2014 winners'] = len(fate)
    f_sk = go.Figure(go.Sankey(arrangement='snap',
                               node=dict(label=[f'{n} ({tot[n]})' for n in nodes], color=[col(n) for n in nodes], pad=18, thickness=18, line_width=0),
                               link=dict(source=src, target=tgt, value=val, color=[col(nodes[t]) + '66' for t in tgt])))
    style(f_sk, h=460, legend=False).update_layout(margin=dict(l=10, r=10, t=10, b=10))
    both = int(((fate.s19 == 'Won') & (fate.s24 == 'Won')).sum())

    return [
        html.H1('Experimental / Cool', className='page-title'),
        section('Money and votes'),
        dbc.Row([
            card('Win rate by asset rank inside the seat', f_r,
                 f'The richest candidate in a seat wins **{wr.iloc[0]:.1f}%** of the time; 6th or lower, **{wr.iloc[-1]:.1f}%**.'),
            card('Assets vs vote share, every candidate', f_vs,
                 f'Money and votes rise together (rank correlation **{rho:.2f}**); the poorest winner declared **{cr(a[a.Winner].Assets_Rs.min())}**.'),
        ]),
        section('Careers'),
        dbc.Row([
            card('Does getting richer win votes?', f_gn,
                 f'No: asset growth barely tracks next-time vote share (correlation **{r_gain:.2f}**).'),
            card('What happened to the 2014 winners?', f_sk,
                 f'Only **{both}** of {len(fate)} 2014 winners won again in 2019 and 2024. Not traced = no linked entry (dropped, moved, or name spelled differently).'),
        ]),
    ]

# ---------------- Data ----------------
def tab_data(states):
    g, a, p = filt(states)
    ss = [s for s in STATES if s in set(g.State)]
    cnt = g.groupby(['Election_Year', 'State']).size().unstack()[ss]
    f_c = go.Figure()
    for y, c in zip(YEARS, ['#86b6ef', '#3987e5', '#184f95']):
        f_c.add_bar(x=ss, y=cnt.loc[y].values, name=str(y), marker_color=c, text=cnt.loc[y].values)
    bar_text(style(f_c)).update_layout(barmode='group')
    f_c.update_yaxes(range=[0, cnt.max().max() * 1.2])

    cov = g.groupby(['State', 'Election_Year']).Has_Assets.mean().unstack().loc[ss] * 100
    f_h = go.Figure(go.Heatmap(z=cov.values, x=[str(y) for y in YEARS], y=ss, colorscale=[[0, '#184f95'], [1, '#86b6ef']], zmin=90, zmax=100,
                               showscale=False, text=[[f'{v:.1f}%' for v in r] for r in cov.values], texttemplate='%{text}',
                               textfont=dict(size=18, color=BG), xgap=3, ygap=3, hovertemplate='%{y} %{x}: %{z:.1f}%<extra></extra>'))
    style(f_h, legend=False).update_yaxes(autorange='reversed')
    f_h.update_xaxes(side='top')
    return [
        html.H1('Data', className='page-title'),
        dbc.Row([
            card('Candidates per state and election', f_c, f'**{len(g):,}** candidate results across **{len(ss)}** states.'),
            card('Share with a matched affidavit (%)', f_h,
                 f'Between **{cov.values.min():.1f}%** and **{cov.values.max():.1f}%** of candidates matched an affidavit with assets.'),
        ]),
    ]


TABS = {'overview': ('Overview', tab_overview), 'wl': ('Winners vs Losers', tab_wl), 'states': ('States', tab_states),
        'parties': ('Parties', tab_parties), 'same': ('Repeat Candidates', tab_same), 'crime': ('Crime', tab_crime),
        'people': ('Education & Age', tab_people), 'women': ('Women', tab_women), 'exp': ('Experimental', tab_exp),
        'data': ('Data', tab_data)}

# quick self-test: build every tab once so errors show up here, not inside the app
for key, (name, fn) in TABS.items():
    fn(STATES)
print('✅ All', len(TABS), 'tabs build')

✅ All 10 tabs build


## Layout
Plain CSES-like page: grey top bar, centred column, uppercase tabs, rows of two `dbc.Col(md=6)` cards on black.

In [16]:
CSS = '''
body{background:#000;color:#fff;font-family:'Helvetica Neue',Arial,sans-serif}
.topbar{background:#2b2b2b;padding:14px 0;margin-bottom:28px}
.topbar .brand{font-weight:800;font-size:1.35rem;letter-spacing:.5px;color:#fff;margin:0}
.wrap{max-width:1320px;margin:0 auto;padding:0 20px}
.page-title{font-weight:800;font-size:2.3rem;margin:1.6rem 0 1.2rem}
.section{font-weight:800;font-size:1.7rem;margin:2rem 0 1rem}
.chart-card{background:#000;border:1px solid #262626;border-radius:6px}
.chart-card .card-title{font-size:1.15rem;font-weight:700}
.cap p{font-size:1.05rem;font-weight:700;margin:.5rem 0 0;color:#fff}
.cap strong{color:#fff}
.lead-text{font-size:1.15rem;font-weight:600;max-width:960px;color:#e6e6e6}
.metric-card{background:#000;border:1px solid #262626;border-radius:6px}
.metric-value{font-size:2rem;font-weight:800;color:#fff}
.metric-label{font-size:.95rem;font-weight:700;color:#b8b8b8}
.form-check-input{background-color:#000;border:1px solid #9a9a9a}
.form-check-input:checked{background-color:#6cb4f8;border-color:#6cb4f8}
.form-select{background-color:#111;color:#fff;border-color:#333;font-weight:600}
.footer{color:#8a8a8a;font-size:.9rem;font-weight:600;border-top:1px solid #333;padding:14px 0 30px;margin-top:20px}
'''

# CSES-style tabs: plain uppercase links, the active one in a grey box, one rule underneath
TAB_STYLE = {'background': 'transparent', 'border': 'none', 'color': '#6cb4f8', 'fontWeight': 500, 'padding': '6px 12px',
             'textTransform': 'uppercase', 'letterSpacing': '.3px', 'fontSize': '15px', 'width': 'auto', 'flex': '0 0 auto'}
TAB_ON = {**TAB_STYLE, 'background': '#3a3a3a', 'color': '#ffffff', 'border': 'none'}

app = Dash(__name__, external_stylesheets=[dbc.themes.DARKLY],
           suppress_callback_exceptions=True, title='Lok Sabha Assets')
app.index_string = app.index_string.replace('{%css%}', '{%css%}<style>' + CSS + '</style>')

app.layout = html.Div([
    html.Div(html.Div(html.P('LOK SABHA ASSETS', className='brand'), className='wrap'), className='topbar'),
    html.Div([
        dcc.Tabs(id='tabs', value='overview', children=[
            dcc.Tab(label=name, value=key, style=TAB_STYLE, selected_style=TAB_ON) for key, (name, _) in TABS.items()
        ], style={'borderBottom': '1px solid #8a8a8a', 'display': 'flex', 'flexWrap': 'wrap', 'gap': '4px'},
           parent_style={'width': '100%'}),
        dcc.Loading(html.Div(id='tab-content'), type='dot', color='#6cb4f8'),
        html.Div('Data: MyNeta affidavits (ADR) · Lok Sabha results via OpenCity / ECI and TCPD · Tamil Nadu, Maharashtra, Bihar, Uttar Pradesh · 2014, 2019, 2024',
                 className='footer'),
    ], className='wrap'),
])

## Callbacks
1. Tabs → tab content.
2. Year radio → wealth histogram. 3. Party dropdown → party box plot. 4. Candidate dropdown → that person's assets.

In [17]:
@app.callback(Output('tab-content', 'children'), Input('tabs', 'value'))
def render_tab(tab):
    return TABS[tab][1](STATES)


@app.callback(Output('hist-graph', 'figure'), Input('hist-year', 'value'))
def update_hist(year):
    return fig_hist(filt(STATES)[1], year)[0]


@app.callback(Output('party-graph', 'figure'), Input('party-pick', 'value'))
def update_party(party):
    return fig_party_box(filt(STATES)[1], party)


@app.callback(Output('person-graph', 'figure'), Input('person-pick', 'value'))
def update_person(pid):
    return fig_person(float(pid))

print('✅ App ready')

✅ App ready


## Run
Shows the dashboard inside the notebook. If the cell area stays blank (VS Code, some JupyterLab setups), use `jupyter_mode='external'` and open the printed link, or `'tab'` to open a browser tab.

In [ ]:
app.run(jupyter_mode='inline', jupyter_height=1100, port=8050)

127.0.0.1 - - [02/Nov/2026 18:59:05] code 400, message Bad request version ('S+ý´j±¤úù\x7fªí\x8e\x00öO\x0f\x05!R\x9cÂu')
127.0.0.1 - - [02/Nov/2026 18:59:05] code 400, message Bad request syntax ('\x16\x03\x01\x05ç\x01\x00\x05ã\x03\x03N=±ðÜÙÆ=ªé`\x8a®Î')
